## Pipeline de dados simples

ETL

In [ ]:
# 1. EXTRAIR (EXTRACT)

import requests
import pandas as pd

# 1. EXTRAÇÃO (EXTRACT)
def extract_pib(codigo=4380,
                data_inicial='01/01/2018',
                data_final='31/12/2025'):

    url = (
        f'https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados'
        f'?formato=json&dataInicial={data_inicial}&dataFinal={data_final}'
    )

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    dados = response.json()
    df_pib = pd.DataFrame(dados)

    return df_pib


# Extração e visualização dos dados originais
df_pib_raw = extract_pib()
display(df_pib_raw)

,data,valor
0,01/01/2018,552484.1
1,01/02/2018,540669.0
2,01/03/2018,588929.9
3,01/04/2018,587220.7
4,01/05/2018,561812.3
...,...,...
91,01/08/2025,1065942.8
92,01/09/2025,1075421.9
93,01/10/2025,1109296.9
94,01/11/2025,1084264.6


In [2]:
# 2. TRANSFORMAÇÃO (TRANSFORM)
def transform_pib(df_pib_raw):

    df_pib = df_pib_raw.copy()

    # Converte a coluna data para datetime
    df_pib['data'] = pd.to_datetime(
        df_pib['data'], format='%d/%m/%Y'
    )

    # Cria a coluna mes_ano no formato mm-aaaa
    df_pib['mes_ano'] = df_pib['data'].dt.strftime('%m-%Y')

    # Converte os valores para números
    df_pib['valor'] = pd.to_numeric(
        df_pib['valor'], errors='coerce'
    )

    # Renomeia a coluna valor
    df_pib = df_pib.rename(
        columns={'valor': 'pib_corrente'}
    )

    # Seleciona as colunas finais
    df_pib_final = df_pib[
        ['data', 'mes_ano', 'pib_corrente']
    ]

    return df_pib_final


# Transformação e visualização dos dados tratados
df_pib = transform_pib(df_pib_raw)
display(df_pib)

,data,mes_ano,pib_corrente
0,2018-01-01,01-2018,552484.1
1,2018-02-01,02-2018,540669.0
2,2018-03-01,03-2018,588929.9
3,2018-04-01,04-2018,587220.7
4,2018-05-01,05-2018,561812.3
...,...,...,...
91,2025-08-01,08-2025,1065942.8
92,2025-09-01,09-2025,1075421.9
93,2025-10-01,10-2025,1109296.9
94,2025-11-01,11-2025,1084264.6


In [3]:
# 3. CARREGAMENTO (LOAD)
def load_pib(df_pib):
    df_pib.to_excel('PIB_Brasileiro.xlsx', index=False)


# Salva a planilha Excel
load_pib(df_pib)